In [11]:
import fastf1
import fastf1.plotting
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import linregress
fastf1.Cache.enable_cache(r"C:\Users\Fatema NGU\Desktop\f1_cache")
fastf1.plotting.setup_mpl(color_scheme='fastf1',misc_mpl_mods=False)
year=2026
gp='Barcelona'
session_type = 'R'
session= fastf1.get_session(year,gp,session_type)
session.load()
laps=session.laps
laps_cleaned=laps.loc[(laps['IsAccurate']==True) & (laps['LapTime'].notna()) & (laps['PitOutTime'].isna()) & (laps['TrackStatus']=='1')].copy()
laps_cleaned['LapTimeSeconds'] = laps_cleaned['LapTime'].dt.total_seconds()
drivers= pd.unique(laps_cleaned['Driver'])
fig, ax = plt.subplots(figsize=(20,10))
compound_colors = {
    compound: fastf1.plotting.get_compound_color(compound, session=session)
    for compound in laps_cleaned['Compound'].unique()
}
degradation_data=[]
for driver in drivers:
    driver_laps=laps_cleaned.loc[laps_cleaned['Driver']==driver]
    stints = driver_laps['Stint'].unique()
    for stint_number in stints:
        stint_laps= driver_laps.loc[driver_laps['Stint']==stint_number].copy()
        if len(stint_laps)<3:
            continue
    compound = stint_laps['Compound'].iloc[0]
    stint_laps['StintLapNumber']=stint_laps['LapNumber']-stint_laps['LapNumber'].min()+1
    slope,intercept,r_value,p_value,std_err = lineregress(stint_laps['StintLapNumber'], stint_laps['LapTimeSeconds'])
    degradation_rate=slope*1000
    degradation_data.append({
        'Driver': driver,
        'Stint': stint_number,
        'Compound': compound,
        'Degradation (ms/lap)': f"{degradation_rate:.1f}"
    })
    ax.plot(stint_laps['StintLapNumber'], stint_laps['LapTimeSeconds'],
           marker='o',linestyle='-',
           color=compound_colors[compound])
    ax.plot(stint_laps['StintLapNumber'], intercept+slope*stint_laps['StintLapNumber'],
           linestyle='--', color=compound_colors[compound],alpha=0.6)
ax.set_title(f'Tire Degradation Analysis - {gp} GP {year}')
ax.set_xlabel('Laps in Stint')
ax.set_ylabel('Lap Time (seconds)')
ax.grid(True,which='both',linestyle='--',linewidth=0.5)
plt.tight_layout()
plt.show()
degradation_df=pd.DataFrame(degradation_data)
print("\nTire Degradation Summary:")
from IPython.display import display
display(degradation_df)


C:\Users\Fatema NGU\AppData\Roaming\Python\Python314\site-packages\fastf1\req.py:251: FutureWarning: `.enable_cache` is deprecated and will be removed in a future version. Use `.configure` instead.
  warnings.warn("`.enable_cache` is deprecated and will be removed in a "
C:\Users\Fatema NGU\AppData\Roaming\Python\Python314\site-packages\fastf1\plotting\_plotting.py:56: FutureWarning: The `misc_mpl_mods` argument was dropped from `.setup_mpl()` in version 3.6.0 and has no effect anymore. It will be removed in a future version of FastF1.
  warnings.warn(
core           INFO 	Loading data for Barcelona Grand Prix - Race [v3.9.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
tokens         INFO 	Signing in to Formula 1 account...
transport    WARNING 	Failed to authenticate with Formula 1 account, continuing without authentication: No F1 account credentials available. Set the environment variables F1_USERNAME 

DataNotLoadedError: The data you are trying to access has not been loaded yet. See `Session.load`